### Docker
Docker = система управления контейнерами

Контейнер = процесс, изолированный по ресурсам и с преднастроенным окружением<br>
Образ (Image) = снепшот файловой системы с записанным на нее приложением, которая монтируется в контейнер

Допустим есть некое приложение app, которое нужно запустить `sh app`. Для универсализации процесса запуска его можно обернуть в контейнер и запускать уже обертку, которая под капотом отвечает за настройку ресурсов `docker run app`

Итого, типовое использование - Docker берет преднастроенный образ 

Почему полезна контейнеризвация:
- Изоляция окружения<br>нет конфликта версий библиотек
- Переносимость (works everywhere)<br>одинаково ставится на разные версии Linux
- Быстрое масштабирование<br>дополнительный экземпляр процесса запускается за секунды
- Изоляция ресурсов<br>контролирование использование
- Повторяемость (reproducibility)<br>экономия на настройке
- Упрощённая доставка (CI/CD)<br>экономия на настройке при развертывании
- Безопасность<br>ограничиваются права, системных вызорвов
- Изоляция сети<br>снаружи видны как отдельные машины

Когда нужно запускать много экземпляров приложения и распределенно, возникает необхоимость делать обертку над оберткой - это уже Kubernetes. Логика та же, но уже в масштабе

Какие конкретно ресурсы ОС изолируются в рамках контейнера?
- с помощью утилиты __namespaces__ выделяется: свое дерево процессов PID, файловая система FS, доменное имя, сетевой менеджемент (iptables etc)
- с помощью утилиты __cgroups__ выделяются свои лимиты на ресурсы (cpu, ram, i/o, network)
- с помощью __OverlayFS__ создается layered файловая системы
- используются инструменты (например, sescomp), чтобы дропунть потенциально опасный функционал

*OverlayedFS - "слоеная" файловая система, когда есть read-only базовый слой, а все модификации наслаиваются в новых слоях. В базовом слое либо голая операционная система (точнее пользовательская ее часть User-space Linux, ядро)

Ядро Linux = драйверы, системные вызовы. Пользоватлеьская часть = shell, более выскоуровневые библиотеки

Можно запускать не только фоне, но и интерактивный процесс (флаг -i -t создает). Можно подключаться к работающему конейтнеру

Что происходит при docker run:
- идет обращение к оркестрирующему процессу __dockerd__, который создает в своей таблице запись про новый контейнер
- форкается новый процесc командой clone (сразу с изоляцией - с созданием новых пространств имен)
- собирается новая файловая система (mount overlay) и её каталог делается корневым (chroot)
- настраиваются лимиты (с помощью namespaces, cgroups, seccomp)
- запуск целевой команды через execve 

### Docker-compose
Docker-compose - более высокоуровневая обертка над Docker. Аналог Kubernetes. Он работает не с отдельными Docker-файлами, а со сложными конфигурациями сервисов, описываемыми в файле конфигурации (yml). Например FastAPI + Triton

В конфигурации указываются ключевые поля: services, image, volumes, ports

<img src="img/docker_compose.png" width=650>

### Docker Swarm

CI/CD регламентирует релиз новых версий приложения
Традиционно функционал CI/CD есть в GitLab
по событию (push, manual run) => сборка + тест + deploy на сервер(а)
Ansible: скрипт для менеджемнта парка машин
Kubernetes: для оркестрации

## Docker command

#### Запуск образа
```
# запустить контейнер
docker run <image>

# дать свое имя
docker run --name <image>

# сделать port forwarding
docker run --p <host_port>:<image_port> <image>

# запустить с добавлением имени контейнера
docker run --hostname <host_name> <image>

# запустить как background-процесс
docker run --d <image>

# запустить образ в интерактивном режиме (с shell)
docker run --it <image>

# подключиться в интерактивном режиме к работающему контейнеру
docker exec -it <container_name_or_id> /bin/bash

```

#### Создание образа

```
docker build

docker build -t <name>

# загрузить образ из репозитория
docker pull
```

#### Мониторинг
```
# вывести все зарегистрированные образы
docker images

# вывести все запущенные на хосте контейнеры
docker ps

# вывести все запущенные контейнером процессы
docker top

# event-log по конетрнеру
docker logs

# накопленная статистика по запущенным контейнера
docker stats
```

#### Управление контейнерами
```
# выгрузить контейнер
docker rm <container>

# запустить контейнер заново
docker start <container>

# поставить выполнение на паузу
docker stop <container>

# выполнить кастомную команду внутри контейнера
docker exec --it <container> <bash commands>

# создать образ
docker commit <container>
```

#### Сохранение образов в Registry
```
# выгрузить образ в репозиторий 
docker push

# созхранить образ в файл
docker save

# загрузить образ из файла
docker load

```

## Registry
Логично сохранять сконфигурированные контейнеры в виде образа. Сохраняются они в библиотеку, которая называется Docker Registry. Есть локальный регистр (образы для использования в рамках одной машины) и глобальный (хранилище для использования на любых машинах)

<img src="img/docker_registry.png" width=500>

## Dockerfile

Dockerfile - это текстовый набор инструкций для развертывания контейнера. В нем указываем какой образ файловой системы подключить

| Инструкция  | Назначение | Пример |
|-------------|------------|--------|
| **FROM**    | Указываем, какой image подключать | `FROM python:3.10` |
| **RUN**     | Кастомные команды ОС для установки пакетов / подготовки окружения | `RUN apt-get update && apt-get install -y git` |
| **COPY**    | Копирует файлы из локальной машины внутрь образа | `COPY app.py /app/app.py` |
| **ADD**     | Как COPY, но умеет распаковывать архивы и скачивать URL | `ADD archive.tar.gz /app/` |
| **WORKDIR** | Устанавливает рабочую директорию для последующих команд чтобы не писать абсолютные пути | `WORKDIR /app` |
| **CMD**     | Стартовая команда (что-то типа void main) | `CMD ["python", "app.py"]` |
| **ENTRYPOINT** | Базовая команда, которую сложно перезаписать; сочетается с CMD | Для обязательной основной логики | `ENTRYPOINT ["python"]` |
| **EXPOSE**  | Указывает, какой порт будет слушать приложение | `EXPOSE 8000` |
| **ENV**     | Устанавливает переменные окружения | `ENV APP_ENV=prod` |
| **VOLUME**  | Какой внешний каталог примонтировать для доступа из образа | `VOLUME /data` |
| **USER**    | Выполняет команды от имени указанного пользователя (для безопасности и изоляции) | `USER appuser` |
| **LABEL**   | Добавляет кастомные метаданные | `LABEL maintainer="you@example.com"` |


Возможно есть некоторая аналогия с CD-дисководом, мы вставляем нужный диск и ОС запускает его

## Полный список команд

#### Основные и самые часто используемые

| Команда | Описание |
|--------|----------|
| **docker run** | Запускает контейнер из образа|
| **docker ps** | Показывает список контейнеров|
| **docker pull** | Загружает образ из registry|
| **docker push** | Загружает образ в registry|
| **docker exec** | Выполняет команду внутри работающего контейнера|
| **docker build** | Собирает образ из Dockerfile|
| **docker image** | Управляет локальными образами|
| **docker images** | Показывает список образов|
| **docker info** | Показывает информацию о системе и демоне Docker|
| **docker version** | Выводит версии Docker|

#### Управление контейнерами
| Команда | Описание |
|--------|----------|
| **docker container** | Команды управления контейнерами (запуск, стоп, rm)|
| **docker inspect** | Показывает детальную информацию об объекте Docker|
| **docker checkpoint** | Создание/восстановление состояния контейнера|
| **docker sandbox** | Создаёт изолированную среду для контейнеров|

#### Управление образами
| Команда | Описание |
|--------|----------|
| **docker buildx** | Расширенная мультиархитектурная сборка|
| **docker builder** | Управляет BuildKit-билдерами|
| **docker manifest** | Управляет multi-arch манифестами образов|
| **docker scout** | Анализ уязвимостей образов|

#### Сети, тома и окружение
| Команда | Описание |
|--------|----------|
| **docker network** | Управляет Docker-сетями|
| **docker volume** | Управляет томами хранения данных|
| **docker context** | Переключение между окружениями Docker|
| **docker system** | Управление ресурсами и очистка|

#### Аутентификация и безопасность
| Команда | Описание |
|--------|----------|
| **docker login** | Авторизация в registry|
| **docker logout** | Выход из registry|
| **docker trust** | Управляет подписью образов|
| **docker secret** | Управляет секретами Swarm|
| **docker pass** | Хранилище секретов через `pass`|
| **docker model** | Управление моделью безопасности (EA)|

#### Docker Compose / Swarm / кластеризация
| Команда | Описание |
|--------|----------|
| **docker compose** | Многоконтейнерные приложения|
| **docker swarm** | Управление кластером Docker Swarm|
| **docker node** | Управляет нодами Swarm|
| **docker stack** | Развёртывание мультисервисных приложений|
| **docker service** | Управляет сервисами в Swarm|
| **docker config** | Конфиги Swarm|

#### Дополнительные и экспериментальные
| Команда | Описание |
|--------|----------|
| **docker search** | Поиск образов в Docker Hub|
| **docker init** | Создаёт базовые Docker-файлы|
| **docker debug** | Отладка контейнеров и окружения|
| **docker offload** | Перенос выполнения контейнеров в облако/удалённый бекенд|
| **docker plugin** | Управление плагинами Docker|
| **docker desktop** | Управление Docker Desktop|
| **docker mcp** | Multi-node Container Platform (эксперимент)|

